# 04 · Análisis comparativo de la contratación por prestación de servicios

**Proyecto SECOP II Barrancabermeja — capa de hallazgos**

Responde los objetivos 3.1 a 3.7, el análisis del ciclo electoral (sección 5) y los rankings
(sección 6) de `docs/proyecto-objetivos.md`.

Hereda de `03` (comparabilidad) y respeta sus dos reglas irrenunciables:

1. **El dinero solo se compara en pesos constantes** (`valor_contrato_real`, `valor_mensual_real`).
2. **Los volúmenes solo se comparan en ventanas alineadas por mes de gobierno.**

### Advertencia metodológica que atraviesa todo el cuaderno

La contratación **no es un flujo mensual continuo: ocurre por tandas**. Ambas administraciones
firman lotes masivos de CPS en pocos días y luego pasan meses casi sin firmar. Por eso:

- Un mes con muchos contratos **no significa** más contratación anual: significa que ahí cayó la tanda.
- Las comparaciones se hacen sobre **agregados de ventana**, nunca mes contra mes.
- Los picos se interpretan como **calendario administrativo**, salvo que haya evidencia de otra cosa.


In [1]:
# 1. Librerías
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

In [2]:
# 2. Rutas
RUTA_ACTUAL = Path.cwd().resolve()
if RUTA_ACTUAL.name.lower() == "notebooks":
    RUTA_PROYECTO = RUTA_ACTUAL.parent
elif (RUTA_ACTUAL / "datos").exists():
    RUTA_PROYECTO = RUTA_ACTUAL
else:
    RUTA_PROYECTO = RUTA_ACTUAL.parent

RUTA_INTERMEDIOS = RUTA_PROYECTO / "datos" / "intermedios"
RUTA_PROCESADOS = RUTA_PROYECTO / "datos" / "procesados"
RUTA_TABLAS = RUTA_PROYECTO / "entregables" / "tablas"
RUTA_TABLAS.mkdir(parents=True, exist_ok=True)

print("Proyecto:", RUTA_PROYECTO)

Proyecto: /home/claude/proj_final


In [3]:
# 3. Cargar la base enriquecida del cuaderno 03

base = pd.read_parquet(RUTA_PROCESADOS / "03_base_analitica_enriquecida.parquet")

COLUMNAS_REQUERIDAS = [
    "alcalde", "mes_gobierno", "anio_gobierno", "valor_contrato_real",
    "valor_mensual_real", "ventana_alineada_gob_16_33", "ventana_ley_garantias",
    "tipo_anio_electoral", "es_cps_estricto", "tipo_cps_claro"
]
faltantes = [c for c in COLUMNAS_REQUERIDAS if c not in base.columns]
if faltantes:
    raise ValueError(
        "Faltan columnas del cuaderno 03: " + ", ".join(faltantes)
        + ". Ejecute 02 y luego 03 antes de este cuaderno."
    )

with open(RUTA_INTERMEDIOS / "03_manifiesto_comparabilidad.json", encoding="utf-8") as f:
    manifiesto_03 = json.load(f)

print("Base heredada:", manifiesto_03.get("version_base"),
      "| que hereda de:", manifiesto_03.get("hereda_de"))
print(f"Contratos: {len(base):,}")

Base heredada: 03_v1 | que hereda de: 02_v5
Contratos: 37,574


In [4]:
# 4. Asegurar los marcadores de calidad del cuaderno 02

# Segun la version del cuaderno 02 con la que se haya construido la base, algunos
# marcadores ya vienen heredados y otros viven solo en el subconjunto CPS.
# Se traen unicamente los que falten, para no duplicar columnas.

MARCADORES = {
    RUTA_PROCESADOS / "02_cps_personas_naturales.parquet": ["flag_valor_mensual_extremo"],
    RUTA_INTERMEDIOS / "02_base_maestra_clasificada.parquet": [
        "flag_duracion_cero", "flag_cps_duracion_atipica", "flag_duracion_no_analizable",
        "flag_valor_mensual_no_confiable", "flag_ejecucion_parcial", "fuente_valor_mensual"
    ]
}

for archivo, columnas in MARCADORES.items():
    faltan = [c for c in columnas if c not in base.columns]
    if not faltan:
        continue
    extra = pd.read_parquet(archivo, columns=["contrato_llave"] + faltan)
    base = base.merge(extra, on="contrato_llave", how="left")

for columna in ["flag_valor_mensual_extremo", "flag_duracion_cero",
                "flag_cps_duracion_atipica", "flag_duracion_no_analizable",
                "flag_valor_mensual_no_confiable", "flag_ejecucion_parcial"]:
    if columna not in base.columns:
        raise ValueError(
            f"Falta el marcador {columna}. Vuelva a ejecutar el cuaderno 02."
        )
    base[columna] = base[columna].fillna(False).astype(bool)

print("Marcadores de calidad disponibles:")
print(f"  Valor mensual extremo        : {int(base['flag_valor_mensual_extremo'].sum()):,}")
print(f"  Duracion cero                : {int(base['flag_duracion_cero'].sum()):,}")
print(f"  CPS con duracion mayor a 12m : {int(base['flag_cps_duracion_atipica'].sum()):,}")
print(f"  Ejecucion parcial (ajustado) : {int(base['flag_ejecucion_parcial'].sum()):,}")
print(f"  Sin base confiable de valor  : {int(base['flag_valor_mensual_no_confiable'].sum()):,}")

Marcadores de calidad disponibles:
  Valor mensual extremo        : 347
  Duracion cero                : 107
  CPS con duracion mayor a 12m : 1
  Ejecucion parcial (ajustado) : 1,458
  Sin base confiable de valor  : 402


## Universo de análisis y marcos de comparación

**Universo principal:** CPS estricto · persona natural · Alcaldía (NIT 890201900) · contrato válido.
La ESE y las demás entidades quedan fuera: no son contratos del alcalde.

**Marcos de comparación:**

| Marco | Uso |
|---|---|
| `ventana_alineada_gob_16_33` | **Comparación principal.** Mismos meses de mandato (16–33) para ambos. |
| Periodo observado completo | Contexto y series de tiempo. **No sirve para comparar volúmenes.** |

**Reglas de exclusión declaradas** (sin imputar nada):

- Valor mensual: se excluyen contratos de **menos de 1 mes** (dividir por una fracción de mes
  infla el equivalente mensual) y los marcados como extremos por el cuaderno 02.
- Duración: se excluyen contratos sin fechas válidas.
- Subtipo profesional/apoyo: solo se compara sobre `tipo_cps_claro`.


In [5]:
# 5. Construir los universos

ALCALDES = ["Alfonso Eljach", "Jonathan Vasquez"]

cps = base[
    (base["es_cps_estricto"] == True)
    & (base["es_alcaldia_analisis"] == True)
    & (base["alcalde"].isin(ALCALDES))
].copy()

# Universo de sensibilidad (criterio mas laxo) para verificar que las conclusiones aguantan
cps_ampliado = base[
    (base["es_cps_ampliado"] == True)
    & (base["es_alcaldia_analisis"] == True)
    & (base["alcalde"].isin(ALCALDES))
].copy()

# Marcos
alineada = cps[cps["ventana_alineada_gob_16_33"]].copy()
N_MESES_ALINEADA = 18

# Subconjunto apto para analisis de valor mensual
apto_valor_mensual = (
    cps["valor_mensual_real"].notna()
    & cps["duracion_meses_exacta"].ge(1)
    & ~cps["flag_valor_mensual_extremo"]
    & ~cps["flag_duracion_no_analizable"]
    & ~cps["flag_valor_mensual_no_confiable"]
)

# Duracion analizable: excluye duracion cero y CPS de mas de 12 meses (errores de captura)
cps["duracion_analizable"] = (
    cps["duracion_meses_exacta"].notna()
    & ~cps["flag_duracion_no_analizable"]
    & ~cps["flag_valor_mensual_no_confiable"]
)
cps["apto_valor_mensual_analisis"] = apto_valor_mensual

print(f"CPS estricto Alcaldía        : {len(cps):,}")
print(f"CPS ampliado (sensibilidad)  : {len(cps_ampliado):,}")
print(f"En ventana alineada (16-33)  : {len(alineada):,}")
print(f"Aptos para valor mensual     : {int(apto_valor_mensual.sum()):,} "
      f"({apto_valor_mensual.mean()*100:.1f}%)")

CPS estricto Alcaldía        : 26,323
CPS ampliado (sensibilidad)  : 26,337
En ventana alineada (16-33)  : 15,610
Aptos para valor mensual     : 24,293 (92.3%)


## Bloque A · Ritmo de contratación: la contratación ocurre por tandas

Antes de comparar nada, hay que entender **cómo** se contrata. La tabla siguiente muestra CPS
firmados por mes y año: los números grandes están concentrados en pocos meses.


In [6]:
# 6. Serie mensual y detección de tandas

serie_mensual = (
    cps.groupby(["anio_periodo", "mes_periodo"])["contrato_llave"]
    .nunique().reset_index(name="cps")
)

tabla_mensual = serie_mensual.pivot(
    index="mes_periodo", columns="anio_periodo", values="cps"
).fillna(0).astype(int)

print("CPS de la Alcaldía firmados por mes y año:")
display(tabla_mensual)

# Una "tanda" = mes que concentra >= 25% de los CPS de su año
totales_anio = serie_mensual.groupby("anio_periodo")["cps"].transform("sum")
serie_mensual["pct_del_anio"] = (serie_mensual["cps"] / totales_anio * 100).round(1)

tandas = (
    serie_mensual[serie_mensual["pct_del_anio"] >= 25]
    .sort_values(["anio_periodo", "mes_periodo"])
)
print("\nTandas de contratación (meses que concentran 25% o más del año):")
display(tandas)

CPS de la Alcaldía firmados por mes y año:


anio_periodo,2021,2022,2023,2024,2025,2026
mes_periodo,,,,,,
1,0,1863,306,345,286,3102
2,0,918,509,480,425,1
3,0,23,379,211,269,0
4,134,0,235,185,285,0
5,395,0,133,607,278,1
6,275,1,2223,157,293,3
7,270,73,0,139,536,286
8,498,503,1,266,380,807
9,574,1428,1,635,775,43



Tandas de contratación (meses que concentran 25% o más del año):


,anio_periodo,mes_periodo,cps,pct_del_anio
9,2022,1,1863,33.6
15,2022,9,1428,25.8
24,2023,6,2223,50.2
54,2026,1,3102,73.1


> **Hallazgo A.** La contratación se concentra en tandas. Esto invalida cualquier lectura
> "mes contra mes" y obliga a comparar por ventanas completas. También explica por qué un mes
> vacío **no** es una caída de la contratación.


In [7]:
# 7. Volumen anual (contexto, no comparación)

resumen_anual = (
    cps.groupby(["alcalde", "anio_periodo"])
    .agg(
        cps=("contrato_llave", "nunique"),
        personas=("proveedor_llave", "nunique"),
        valor_real_total=("valor_contrato_real", "sum"),
        duracion_mediana=("duracion_meses_exacta", "median")
    )
    .reset_index()
)
resumen_anual["contratos_por_persona"] = (
    resumen_anual["cps"] / resumen_anual["personas"]
).round(2)

print("Volumen por año (2026 va hasta el corte del 6 de septiembre):")
display(resumen_anual.round(2))

Volumen por año (2026 va hasta el corte del 6 de septiembre):


,alcalde,anio_periodo,cps,personas,valor_real_total,duracion_mediana,contratos_por_persona
0,Alfonso Eljach,2021,2840,2266,3.312103e+10,2.99,1.25
1,Alfonso Eljach,2022,5540,3544,7.459745e+10,3.71,1.56
2,Alfonso Eljach,2023,4430,3079,6.427697e+10,4.01,1.44
3,Jonathan Vasquez,2024,4572,2672,5.098128e+10,2.92,1.71
4,Jonathan Vasquez,2025,4698,3293,5.885609e+10,3.12,1.43
5,Jonathan Vasquez,2026,4243,3192,5.648884e+10,3.91,1.33


## Bloque B · Volumen y personas — objetivos 3.1 y 3.2

Comparación **en la ventana alineada** (meses de gobierno 16–33 de cada mandato).


In [8]:
# 8. Indicadores de volumen y personas en la ventana alineada

personas_alineada = (
    alineada.groupby(["alcalde", "proveedor_llave"])
    .agg(
        contratos=("contrato_llave", "nunique"),
        valor_real=("valor_contrato_real", "sum"),
        meses_contratados=("duracion_meses_exacta", "sum")
    )
    .reset_index()
)

resumen_b = (
    personas_alineada.groupby("alcalde")
    .agg(
        personas_unicas=("proveedor_llave", "nunique"),
        contratos=("contratos", "sum"),
        contratos_por_persona=("contratos", "mean"),
        valor_real_total=("valor_real", "sum")
    )
    .reset_index()
)
resumen_b["cps_por_mes"] = (resumen_b["contratos"] / N_MESES_ALINEADA).round(1)
resumen_b["personas_por_mes"] = (resumen_b["personas_unicas"] / N_MESES_ALINEADA).round(1)
resumen_b["valor_real_por_mes"] = (
    resumen_b["valor_real_total"] / N_MESES_ALINEADA
).round(0)

print(f"Ventana alineada · meses de gobierno 16-33 ({N_MESES_ALINEADA} meses por alcalde)")
print("Valores en pesos constantes de 2025")
display(resumen_b.round(2))

Ventana alineada · meses de gobierno 16-33 (18 meses por alcalde)
Valores en pesos constantes de 2025


,alcalde,personas_unicas,contratos,contratos_por_persona,valor_real_total,cps_por_mes,personas_por_mes,valor_real_por_mes
0,Alfonso Eljach,3677,7649,2.08,1.035675e+11,424.9,204.3,5.753750e+09
1,Jonathan Vasquez,4348,7961,1.83,9.604962e+10,442.3,241.6,5.336090e+09


## Bloque C · Duración de los contratos — objetivo 3.3

**Exclusiones declaradas.** Se dejan fuera de las estadísticas de duración los contratos con
**duración cero** (la fecha de fin no fue diligenciada y quedó igual a la de inicio) y los **CPS
de más de 12 meses** (atípicos para prestación de servicios; el caso detectado tiene fecha de fin
en 2029 para un contrato firmado en 2026, lo que apunta a un error de digitación del año).

Son errores de captura de SECOP, no contratos reales. Siguiendo la política del proyecto **no se
imputan ni se borran**: quedan marcados en la base y auditados en
`datos/intermedios/02_revision_duracion_atipica.csv`.


In [9]:
# 9. Duración: percentiles y rangos definidos en los objetivos

# Se reconstruye la ventana alineada sobre "cps" para heredar duracion_analizable
alineada = cps[cps["ventana_alineada_gob_16_33"]].copy()

con_duracion = alineada["duracion_meses_exacta"].notna()
excluidos = int((con_duracion & ~alineada["duracion_analizable"]).sum())

dur = alineada[alineada["duracion_analizable"]].copy()

print(f"Contratos excluidos del analisis de duracion por error de captura: {excluidos}")
print("(duracion cero o CPS de mas de 12 meses; se conservan en la base y se auditan")
print(" en datos/intermedios/02_revision_duracion_atipica.csv)")
print()

percentiles_duracion = (
    dur.groupby("alcalde")["duracion_meses_exacta"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
    .round(2)
)
print("Duración en meses (ventana alineada):")
display(percentiles_duracion)


def rango_duracion(meses):
    if pd.isna(meses):
        return "Sin dato"
    if meses <= 3:
        return "Hasta 3 meses"
    if meses <= 4:
        return "Mas de 3 y hasta 4"
    if meses <= 6:
        return "Mas de 4 y hasta 6"
    if meses <= 9:
        return "Mas de 6 y hasta 9"
    if meses <= 12:
        return "Mas de 9 y hasta 12"
    return "Mas de 12 meses"


ORDEN_RANGOS = [
    "Hasta 3 meses", "Mas de 3 y hasta 4", "Mas de 4 y hasta 6",
    "Mas de 6 y hasta 9", "Mas de 9 y hasta 12", "Mas de 12 meses"
]

dur["rango_duracion"] = pd.Categorical(
    dur["duracion_meses_exacta"].map(rango_duracion),
    categories=ORDEN_RANGOS, ordered=True
)

distribucion_duracion = (
    dur.groupby(["alcalde", "rango_duracion"], observed=True)["contrato_llave"]
    .nunique().reset_index(name="contratos")
)
distribucion_duracion["porcentaje"] = (
    distribucion_duracion["contratos"]
    / distribucion_duracion.groupby("alcalde")["contratos"].transform("sum") * 100
).round(1)

print("\nDistribución por rangos de duración:")
display(
    distribucion_duracion.pivot(
        index="rango_duracion", columns="alcalde", values="porcentaje"
    ).fillna(0)
)

Contratos excluidos del analisis de duracion por error de captura: 74
(duracion cero o CPS de mas de 12 meses; se conservan en la base y se auditan
 en datos/intermedios/02_revision_duracion_atipica.csv)

Duración en meses (ventana alineada):


,count,mean,std,min,25%,50%,75%,90%,max
alcalde,,,,,,,,,
Alfonso Eljach,7546.0,3.57,1.28,0.03,2.96,3.45,4.24,5.01,10.41
Jonathan Vasquez,7764.0,3.54,1.33,0.13,2.92,3.45,4.01,5.35,10.55



Distribución por rangos de duración:


alcalde,Alfonso Eljach,Jonathan Vasquez
rango_duracion,,
Hasta 3 meses,43.9,48.8
Mas de 3 y hasta 4,27.8,23.2
Mas de 4 y hasta 6,27.5,25.4
Mas de 6 y hasta 9,0.8,2.4
Mas de 9 y hasta 12,0.1,0.2


## Bloque D · Valores y valor mensual equivalente — objetivo 3.4

**Todo en pesos constantes de 2025.** Se reporta **mediana** (no promedio): la distribución es
asimétrica y el promedio se deja arrastrar por unos pocos contratos grandes.

**Corrección crítica aplicada en el cuaderno 02.** SECOP registra la fecha de fin **real**, así
que un contrato terminado anticipadamente queda con duración corta pero conserva el valor total
pactado. Dividir uno entre otro infla el valor mensual hasta 3 veces. Cuando el contrato ya
terminó y quedó más del 5 % sin ejecutar, el valor mensual se calcula sobre el **valor
efectivamente ejecutado**, que corresponde al mismo periodo que la duración registrada.

**Exclusiones declaradas:** contratos de menos de un mes, valores extremos marcados por el
cuaderno 02, errores de captura de fechas, y contratos terminados sin ningún valor ejecutado
reportado (no hay base para estimar el valor mensual).


In [10]:
# 10. Valor mensual equivalente real, con intervalo de confianza por bootstrap

vm = alineada[alineada["apto_valor_mensual_analisis"]].copy()

excluidos_valor = len(alineada) - len(vm)
print(f"Contratos excluidos del analisis de valor: {excluidos_valor:,} "
      f"({excluidos_valor/len(alineada)*100:.1f}%)")
print("(duracion menor a 1 mes, valores extremos, errores de captura o")
print(" contratos terminados sin valor ejecutado reportado)")
print()

resumen_valores = (
    vm.groupby("alcalde")
    .agg(
        contratos=("contrato_llave", "nunique"),
        vm_real_mediano=("valor_mensual_real", "median"),
        vm_real_p25=("valor_mensual_real", lambda x: x.quantile(0.25)),
        vm_real_p75=("valor_mensual_real", lambda x: x.quantile(0.75)),
        valor_contrato_real_mediano=("valor_contrato_real", "median")
    )
    .reset_index()
)
print("Valor mensual equivalente en pesos constantes de 2025 (ventana alineada):")
display(resumen_valores.round(0))


def ic_bootstrap_mediana(valores, n_iter=2000, semilla=42):
    """Intervalo de confianza del 95% para la mediana, por bootstrap."""
    rng = np.random.default_rng(semilla)
    valores = np.asarray(valores.dropna())
    if len(valores) < 30:
        return (np.nan, np.nan)
    medianas = [
        np.median(rng.choice(valores, size=len(valores), replace=True))
        for _ in range(n_iter)
    ]
    return tuple(np.percentile(medianas, [2.5, 97.5]))


print("\nIntervalos de confianza del 95% para la mediana (bootstrap, 2.000 réplicas):")
for alcalde in ALCALDES:
    serie = vm.loc[vm["alcalde"] == alcalde, "valor_mensual_real"]
    bajo, alto = ic_bootstrap_mediana(serie)
    print(f"  {alcalde:20} mediana {serie.median():>12,.0f}   IC95% [{bajo:,.0f} - {alto:,.0f}]")

print("\nSi los intervalos NO se traslapan, la diferencia es estadísticamente sólida.")

Contratos excluidos del analisis de valor: 893 (5.7%)
(duracion menor a 1 mes, valores extremos, errores de captura o
 contratos terminados sin valor ejecutado reportado)

Valor mensual equivalente en pesos constantes de 2025 (ventana alineada):


,alcalde,contratos,vm_real_mediano,vm_real_p25,vm_real_p75,valor_contrato_real_mediano
0,Alfonso Eljach,7224,3427694.0,2467424.0,4546214.0,11826106.0
1,Jonathan Vasquez,7493,2940873.0,2494954.0,3976670.0,10816272.0



Intervalos de confianza del 95% para la mediana (bootstrap, 2.000 réplicas):


  Alfonso Eljach       mediana    3,427,694   IC95% [3,347,333 - 3,481,227]


  Jonathan Vasquez     mediana    2,940,873   IC95% [2,854,072 - 2,982,502]

Si los intervalos NO se traslapan, la diferencia es estadísticamente sólida.


## Bloque E · Recurrencia de contratistas — objetivo 3.5

Se crean aquí las variables derivadas que pedían los objetivos (sección 8):
`numero_contratos_persona`, `contrato_numero_persona`, `es_recontratacion`.
Se calculan **dentro de cada administración**, que es la unidad de análisis del objetivo.


In [11]:
# 11. Variables derivadas de recurrencia

cps = cps.sort_values(["alcalde", "proveedor_llave", "fecha_asignacion_periodo"])

cps["numero_contratos_persona"] = (
    cps.groupby(["alcalde", "proveedor_llave"])["contrato_llave"].transform("nunique")
)
cps["contrato_numero_persona"] = (
    cps.groupby(["alcalde", "proveedor_llave"]).cumcount() + 1
)
cps["es_recontratacion"] = cps["contrato_numero_persona"] > 1

print("Variables de recurrencia creadas.")
print(f"Contratos que son recontratación: {int(cps['es_recontratacion'].sum()):,} "
      f"({cps['es_recontratacion'].mean()*100:.1f}%)")

Variables de recurrencia creadas.
Contratos que son recontratación: 16,010 (60.8%)


In [12]:
# 12. Distribución de la recurrencia en la ventana alineada

rec = (
    cps[cps["ventana_alineada_gob_16_33"]]
    .groupby(["alcalde", "proveedor_llave"])["contrato_llave"]
    .nunique().reset_index(name="contratos")
)


def banda_recurrencia(n):
    if n == 1:
        return "1 contrato"
    if n == 2:
        return "2 contratos"
    if n in (3, 4):
        return "3 a 4 contratos"
    return "5 o mas"


ORDEN_REC = ["1 contrato", "2 contratos", "3 a 4 contratos", "5 o mas"]
rec["banda"] = pd.Categorical(
    rec["contratos"].map(banda_recurrencia), categories=ORDEN_REC, ordered=True
)

distribucion_recurrencia = (
    rec.groupby(["alcalde", "banda"], observed=True)["proveedor_llave"]
    .nunique().reset_index(name="personas")
)
distribucion_recurrencia["porcentaje"] = (
    distribucion_recurrencia["personas"]
    / distribucion_recurrencia.groupby("alcalde")["personas"].transform("sum") * 100
).round(1)

print("Distribución de personas por número de contratos (ventana alineada):")
display(
    distribucion_recurrencia.pivot(
        index="banda", columns="alcalde", values="porcentaje"
    ).fillna(0)
)

resumen_recurrencia = (
    rec.groupby("alcalde")
    .agg(
        personas=("proveedor_llave", "nunique"),
        contratos_mediana=("contratos", "median"),
        contratos_max=("contratos", "max"),
        pct_2_o_mas=("contratos", lambda x: (x >= 2).mean() * 100),
        pct_3_o_mas=("contratos", lambda x: (x >= 3).mean() * 100),
        pct_5_o_mas=("contratos", lambda x: (x >= 5).mean() * 100)
    )
    .reset_index()
)
print("\nIndicadores de recurrencia:")
display(resumen_recurrencia.round(2))

Distribución de personas por número de contratos (ventana alineada):


alcalde,Alfonso Eljach,Jonathan Vasquez
banda,,
1 contrato,41.3,49.1
2 contratos,22.6,24.3
3 a 4 contratos,34.0,26.2
5 o mas,2.1,0.3



Indicadores de recurrencia:


,alcalde,personas,contratos_mediana,contratos_max,pct_2_o_mas,pct_3_o_mas,pct_5_o_mas
0,Alfonso Eljach,3677,2.0,8,58.69,36.12,2.12
1,Jonathan Vasquez,4348,2.0,7,50.92,26.59,0.34


## Bloque F · Renovaciones y continuidad — objetivo 3.6

Se mide el tiempo entre el **fin de un contrato** y el **inicio del siguiente** de la misma persona
dentro de la misma administración.

**Decisión metodológica declarada:** cuando el siguiente contrato empieza *antes* de que termine
el anterior, no es una renovación sino un **contrato simultáneo**. Se clasifica aparte en lugar
de contarlo como "renovación de 0 días", que inflaría artificialmente la banda de continuidad
inmediata.


In [13]:
# 13. Días entre contratos consecutivos de la misma persona

ren = cps.dropna(
    subset=["fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato"]
).sort_values(["alcalde", "proveedor_llave", "fecha_de_inicio_del_contrato"]).copy()

ren["fin_contrato_anterior"] = (
    ren.groupby(["alcalde", "proveedor_llave"])["fecha_de_fin_del_contrato"].shift()
)
ren["dias_desde_contrato_anterior"] = (
    ren["fecha_de_inicio_del_contrato"] - ren["fin_contrato_anterior"]
).dt.days


def banda_renovacion(dias):
    if pd.isna(dias):
        return "Primer contrato"
    if dias < 0:
        return "Contrato simultaneo (solapado)"
    if dias <= 7:
        return "0 a 7 dias"
    if dias <= 30:
        return "8 a 30 dias"
    if dias <= 90:
        return "31 a 90 dias"
    return "Mas de 90 dias"


ORDEN_RENOV = [
    "Contrato simultaneo (solapado)", "0 a 7 dias", "8 a 30 dias",
    "31 a 90 dias", "Mas de 90 dias", "Primer contrato"
]
ren["banda_renovacion"] = pd.Categorical(
    ren["dias_desde_contrato_anterior"].map(banda_renovacion),
    categories=ORDEN_RENOV, ordered=True
)

renovaciones = ren[ren["banda_renovacion"].ne("Primer contrato")]

distribucion_renovaciones = (
    renovaciones.groupby(["alcalde", "banda_renovacion"], observed=True)["contrato_llave"]
    .nunique().reset_index(name="contratos")
)
distribucion_renovaciones["porcentaje"] = (
    distribucion_renovaciones["contratos"]
    / distribucion_renovaciones.groupby("alcalde")["contratos"].transform("sum") * 100
).round(1)

print("Tiempo entre el fin de un contrato y el inicio del siguiente (misma persona):")
display(
    distribucion_renovaciones.pivot(
        index="banda_renovacion", columns="alcalde", values="porcentaje"
    ).fillna(0)
)

print("\nMediana de días entre contratos (excluyendo simultáneos):")
print(
    renovaciones.loc[renovaciones["dias_desde_contrato_anterior"].ge(0)]
    .groupby("alcalde")["dias_desde_contrato_anterior"].median()
)

Tiempo entre el fin de un contrato y el inicio del siguiente (misma persona):


alcalde,Alfonso Eljach,Jonathan Vasquez
banda_renovacion,,
Contrato simultaneo (solapado),2.3,2.4
0 a 7 dias,4.6,5.8
8 a 30 dias,24.6,34.2
31 a 90 dias,41.3,39.1
Mas de 90 dias,27.2,18.5



Mediana de días entre contratos (excluyendo simultáneos):
alcalde
Alfonso Eljach      48.0
Jonathan Vasquez    36.0
Name: dias_desde_contrato_anterior, dtype: float64


## Bloque G · Contratistas compartidos entre gobiernos — objetivo 3.7

Personas naturales que recibieron CPS de la Alcaldía en **ambas** administraciones. La llave es
el **documento normalizado**, no el nombre.


In [14]:
# 14. Contratistas presentes en las dos administraciones

por_alcalde = (
    cps.groupby(["proveedor_llave", "alcalde"])
    .agg(
        contratos=("contrato_llave", "nunique"),
        valor_real=("valor_contrato_real", "sum"),
        meses=("duracion_meses_exacta", "sum"),
        primer_contrato=("fecha_asignacion_periodo", "min"),
        ultimo_contrato=("fecha_asignacion_periodo", "max")
    )
    .reset_index()
)

pivote = por_alcalde.pivot(index="proveedor_llave", columns="alcalde", values="contratos")
compartidos = pivote.dropna().index

cps["contratista_compartido"] = cps["proveedor_llave"].isin(compartidos)

personas_alfonso = set(por_alcalde.loc[por_alcalde["alcalde"] == "Alfonso Eljach", "proveedor_llave"])
personas_jonathan = set(por_alcalde.loc[por_alcalde["alcalde"] == "Jonathan Vasquez", "proveedor_llave"])

print(f"Personas con CPS en la administración de Alfonso : {len(personas_alfonso):,}")
print(f"Personas con CPS en la administración de Jonathan: {len(personas_jonathan):,}")
print(f"Personas presentes en AMBAS                      : {len(compartidos):,}")
print()
print(f"Continuidad: {len(compartidos)/len(personas_alfonso)*100:.1f}% de los contratistas "
      f"de Alfonso siguieron contratando con Jonathan")
print(f"Renovación : {(1 - len(compartidos)/len(personas_jonathan))*100:.1f}% de los contratistas "
      f"de Jonathan son nuevos (no venían de Alfonso)")

detalle_compartidos = (
    por_alcalde[por_alcalde["proveedor_llave"].isin(compartidos)]
    .pivot(index="proveedor_llave", columns="alcalde",
           values=["contratos", "valor_real", "meses"])
)
detalle_compartidos.columns = ["_".join(c).strip() for c in detalle_compartidos.columns]
detalle_compartidos = detalle_compartidos.reset_index()

nombres = cps.drop_duplicates("proveedor_llave")[
    ["proveedor_llave", "proveedor_nombre_canonico"]
]
detalle_compartidos = detalle_compartidos.merge(nombres, on="proveedor_llave", how="left")

print("\nResumen agregado de los contratistas compartidos (pesos constantes):")
display(
    detalle_compartidos[[c for c in detalle_compartidos.columns if c != "proveedor_llave"]]
    .select_dtypes(include=[np.number]).sum().round(0)
)

Personas con CPS en la administración de Alfonso : 4,995
Personas con CPS en la administración de Jonathan: 5,318
Personas presentes en AMBAS                      : 1,589

Continuidad: 31.8% de los contratistas de Alfonso siguieron contratando con Jonathan
Renovación : 70.1% de los contratistas de Jonathan son nuevos (no venían de Alfonso)

Resumen agregado de los contratistas compartidos (pesos constantes):


contratos_Alfonso Eljach       5.080000e+03
contratos_Jonathan Vasquez     5.305000e+03
valor_real_Alfonso Eljach      7.412452e+10
valor_real_Jonathan Vasquez    7.452054e+10
meses_Alfonso Eljach           1.920400e+04
meses_Jonathan Vasquez         1.911600e+04
dtype: float64

## Bloque H · Ciclo electoral y ley de garantías — sección 5

Este es el bloque de mayor interés público. La pregunta no es si hubo contratación en la ventana
restringida (casi no la hubo, como manda la norma), sino **qué pasó justo antes de que empezara**.


In [15]:
# 15. Comportamiento alrededor de la ventana de ley de garantías (elección 29-oct-2023)

ELECCION_2023 = pd.Timestamp("2023-10-29")
INICIO_VENTANA_2023 = ELECCION_2023 - pd.DateOffset(months=4)   # 29-jun-2023

print(f"Ventana de restricción: {INICIO_VENTANA_2023.date()} a {ELECCION_2023.date()}")
print()

# Firmas por dia en las tres semanas previas al cierre
previo = cps[
    cps["fecha_asignacion_periodo"].between(
        INICIO_VENTANA_2023 - pd.Timedelta(days=21), ELECCION_2023
    )
]
firmas_dia = (
    previo.groupby(previo["fecha_asignacion_periodo"].dt.date)["contrato_llave"]
    .nunique()
)
print("CPS firmados por día en las 3 semanas previas al cierre de la ventana:")
print(firmas_dia.to_string())

# Contratos vigentes durante la ventana y de donde vienen
vigentes = cps[
    (cps["fecha_de_inicio_del_contrato"] <= ELECCION_2023)
    & (cps["fecha_de_fin_del_contrato"] >= INICIO_VENTANA_2023)
]
firmados_en_carrera = vigentes[
    vigentes["fecha_asignacion_periodo"].between(
        INICIO_VENTANA_2023 - pd.Timedelta(days=30), INICIO_VENTANA_2023
    )
]

print()
print(f"CPS vigentes durante la ventana de restricción : {vigentes['contrato_llave'].nunique():,}")
print(f"Personas cubiertas                            : {vigentes['proveedor_llave'].nunique():,}")
print(f"De ellos, firmados en los 30 días previos      : {firmados_en_carrera['contrato_llave'].nunique():,}"
      f" ({firmados_en_carrera['contrato_llave'].nunique()/vigentes['contrato_llave'].nunique()*100:.0f}%)")
print()
print("Mes en que terminan los contratos de esa carrera previa:")
print(
    firmados_en_carrera["fecha_de_fin_del_contrato"]
    .dt.to_period("M").astype(str).value_counts().sort_index().to_string()
)

Ventana de restricción: 2023-06-29 a 2023-10-29



CPS firmados por día en las 3 semanas previas al cierre de la ventana:
fecha_asignacion_periodo
2023-06-08     25
2023-06-09     52
2023-06-10     31
2023-06-12     19
2023-06-13     43
2023-06-14     39
2023-06-15     53
2023-06-16     61
2023-06-17     72
2023-06-19     35
2023-06-20    119
2023-06-21    105
2023-06-22    153
2023-06-23    209
2023-06-24    234
2023-06-25      1
2023-06-26    322
2023-06-27    330
2023-06-28    288
2023-08-30      1
2023-09-28      1
2023-10-12      1
2023-10-26      2

CPS vigentes durante la ventana de restricción : 2,866
Personas cubiertas                            : 2,804
De ellos, firmados en los 30 días previos      : 2,221 (77%)

Mes en que terminan los contratos de esa carrera previa:
fecha_de_fin_del_contrato
2023-07       4
2023-08      20
2023-09     159
2023-10     382
2023-11     596
2023-12    1060


> **Hallazgo H (el central del proyecto).** La administración saliente firmó una tanda masiva de
> CPS en los **últimos diez días hábiles antes de que empezara la restricción**, con duraciones
> calibradas para terminar en **noviembre y diciembre de 2023**, es decir, cubriendo toda la
> ventana electoral y el cierre del mandato.
>
> **Cómo debe comunicarse.** Contratar antes de que empiece la restricción **es legal**. El
> hallazgo es un **patrón de anticipación al calendario legal**, no una irregularidad. Siguiendo
> la escala del proyecto (objetivos, sección 11), esto es un *patrón relevante* que puede motivar
> revisión documental — nunca una acusación derivada de una estadística.


In [16]:
# 16. Contratación por tipo de año electoral (normalizada por meses TRANSCURRIDOS)

# IMPORTANTE: la tasa se divide por los meses realmente transcurridos dentro de la
# cobertura confiable, NO por los meses que tuvieron contratos. Como la contratación
# es por tandas, dividir por "meses con contratos" infla los años con vacíos.

INICIO_COBERTURA = pd.Timestamp("2021-04-01")   # cobertura confiable de la Alcaldía
CORTE_DATOS = pd.Timestamp("2026-09-06")


def meses_transcurridos(anio):
    inicio = max(pd.Timestamp(f"{anio}-01-01"), INICIO_COBERTURA)
    fin = min(pd.Timestamp(f"{anio}-12-31"), CORTE_DATOS)
    if fin <= inicio:
        return np.nan
    return (fin - inicio).days / 30.44


electoral = (
    cps.groupby(["alcalde", "anio_periodo", "tipo_anio_electoral"])["contrato_llave"]
    .nunique().reset_index(name="cps")
)
electoral["meses_transcurridos"] = (
    electoral["anio_periodo"].map(meses_transcurridos).round(1)
)
electoral["cps_por_mes"] = (
    electoral["cps"] / electoral["meses_transcurridos"]
).round(1)
electoral["anio_parcial"] = electoral["anio_periodo"].isin([2021, 2026])

print("Contratación por año y tipo de año electoral")
print("(tasa sobre meses transcurridos; 2021 y 2026 son años parciales)")
display(electoral)

print("\nPromedio de CPS por mes según el tipo de año:")
display(
    electoral.groupby(["alcalde", "tipo_anio_electoral"])["cps_por_mes"]
    .mean().round(1).reset_index()
)

Contratación por año y tipo de año electoral
(tasa sobre meses transcurridos; 2021 y 2026 son años parciales)


,alcalde,anio_periodo,tipo_anio_electoral,cps,meses_transcurridos,cps_por_mes,anio_parcial
0,Alfonso Eljach,2021,Ordinario,2840,9.0,315.6,True
1,Alfonso Eljach,2022,Preelectoral,5540,12.0,461.7,False
2,Alfonso Eljach,2023,Electoral,4430,12.0,369.2,False
3,Jonathan Vasquez,2024,Ordinario,4572,12.0,381.0,False
4,Jonathan Vasquez,2025,Ordinario,4698,12.0,391.5,False
5,Jonathan Vasquez,2026,Preelectoral,4243,8.1,523.8,True



Promedio de CPS por mes según el tipo de año:


,alcalde,tipo_anio_electoral,cps_por_mes
0,Alfonso Eljach,Electoral,369.2
1,Alfonso Eljach,Ordinario,315.6
2,Alfonso Eljach,Preelectoral,461.7
3,Jonathan Vasquez,Ordinario,386.2
4,Jonathan Vasquez,Preelectoral,523.8


## Bloque I · Rankings de contratistas — sección 6

Los rankings usan **pesos constantes** y excluyen contratos de menos de un mes en las métricas
de valor mensual.

> **Advertencia para publicación.** Aparecer en un ranking **no implica irregularidad alguna**.
> Son contratos públicos y legales. Antes de publicar cualquier nombre, verificar el caso contra
> el expediente en SECOP (columna `urlproceso`) y aplicar la escala de evidencia del proyecto.


In [17]:
# 17. Rankings

def top(df, columna, n=15):
    return df.nlargest(n, columna).reset_index(drop=True)


ranking_personas = (
    cps.groupby(["proveedor_llave", "proveedor_nombre_canonico"])
    .agg(
        contratos=("contrato_llave", "nunique"),
        valor_real_acumulado=("valor_contrato_real", "sum"),
        meses_acumulados=("duracion_meses_exacta", "sum"),
        primer_contrato=("fecha_asignacion_periodo", "min"),
        ultimo_contrato=("fecha_asignacion_periodo", "max"),
        administraciones=("alcalde", "nunique")
    )
    .reset_index()
)
ranking_personas["anios_distintos"] = (
    cps.groupby("proveedor_llave")["anio_periodo"].nunique().values
)

print("TOP 15 · Por número de contratos:")
display(top(ranking_personas, "contratos")[
    ["proveedor_nombre_canonico", "contratos", "valor_real_acumulado",
     "meses_acumulados", "administraciones"]
].round(0))

print("\nTOP 15 · Por valor acumulado en pesos constantes:")
display(top(ranking_personas, "valor_real_acumulado")[
    ["proveedor_nombre_canonico", "valor_real_acumulado", "contratos",
     "meses_acumulados", "administraciones"]
].round(0))

print("\nTOP 15 · Por permanencia (meses acumulados contratados):")
display(top(ranking_personas, "meses_acumulados")[
    ["proveedor_nombre_canonico", "meses_acumulados", "contratos",
     "anios_distintos", "administraciones"]
].round(1))

# Contratos individuales con mayor valor mensual real
ranking_contratos_vm = (
    cps[cps["apto_valor_mensual_analisis"]]
    .nlargest(15, "valor_mensual_real")
    [["proveedor_nombre_canonico", "alcalde", "referencia_del_contrato",
      "valor_mensual_real", "valor_contrato_real", "valor_ejecutado_num",
      "pct_ejecucion", "duracion_meses_exacta", "fuente_valor_mensual",
      "tipo_cps", "urlproceso"]]
    .reset_index(drop=True)
)
print("\nTOP 15 · Contratos con mayor valor mensual equivalente (pesos constantes):")
display(ranking_contratos_vm.round(0))

TOP 15 · Por número de contratos:


,proveedor_nombre_canonico,contratos,valor_real_acumulado,meses_acumulados,administraciones
0,Liliana Lopez,20,237666775.0,92.0,2
1,EMILIO RAFAEL HERAZO ISAZA,19,327160938.0,65.0,2
2,DIANA PATRICIA BENAVIDEZ,16,279382456.0,56.0,2
3,MARYERLI MILENA CASTRO BAUTISTA,15,202461890.0,49.0,2
4,libardo corzo colon,15,241861303.0,74.0,2
5,MARIA FERNANDA OJEDA AMADO,15,321856739.0,55.0,2
6,YAIRCIÑO AMARIS MONSALVE,15,176875499.0,62.0,2
7,evelyn del carmen collazos,15,399399466.0,68.0,2
8,ROSALBA CEPEDA ROJAS,15,264947181.0,63.0,2
9,ROCIO DEL CARMEN DIAZ SARMIENTO,15,163825244.0,42.0,2



TOP 15 · Por valor acumulado en pesos constantes:


,proveedor_nombre_canonico,valor_real_acumulado,contratos,meses_acumulados,administraciones
0,Carlos Ariel Urzola Núñez,475789828.0,13,60.0,2
1,Edwin De la Cruz,466090611.0,13,59.0,2
2,JAIME ANTONIO ROJAS CARCAMO,408649475.0,13,62.0,2
3,ANGELICA MARIA MARTINEZ ROJAS,404143455.0,12,53.0,2
4,evelyn del carmen collazos,399399466.0,15,68.0,2
5,Olga Patricia Márquez Calderón,392976869.0,12,55.0,2
6,HEBERT JOSE ESPELETA TINOCO,378773058.0,13,59.0,2
7,MERCEDES CECILIA VASQUEZ VIDALES,368694123.0,15,62.0,2
8,Andres Mauricio Hernandez C,364748734.0,12,54.0,2
9,EDD RUIZ,363602892.0,11,51.0,2



TOP 15 · Por permanencia (meses acumulados contratados):


,proveedor_nombre_canonico,meses_acumulados,contratos,anios_distintos,administraciones
0,Liliana Lopez,92.2,20,6,2
1,libardo corzo colon,74.4,15,6,2
2,evelyn del carmen collazos,68.3,15,6,2
3,EMILIO RAFAEL HERAZO ISAZA,64.7,19,6,2
4,ROSALBA CEPEDA ROJAS,62.7,15,6,2
5,MERCEDES CECILIA VASQUEZ VIDALES,62.0,15,6,2
6,YAIRCIÑO AMARIS MONSALVE,61.9,15,6,2
7,JAIME ANTONIO ROJAS CARCAMO,61.8,13,6,2
8,VICTOR JULIAN GOMEZ TARAZONA,60.3,13,6,2
9,MENDEZ VALENCIA RAQUEL,60.2,13,6,2



TOP 15 · Contratos con mayor valor mensual equivalente (pesos constantes):


,proveedor_nombre_canonico,alcalde,referencia_del_contrato,valor_mensual_real,valor_contrato_real,valor_ejecutado_num,pct_ejecucion,duracion_meses_exacta,fuente_valor_mensual,tipo_cps,urlproceso
0,sandra milena jimenez oliveros,Alfonso Eljach,CONTRATO 3119-21,10178348.0,11368720.0,8450000,1.0,1.0,Valor del contrato,Profesional claro,{'url': 'https://community.secop.gov.co/Public...
1,Jhon Ospino,Alfonso Eljach,CONTRATO 0953-21,9776210.0,19269797.0,14000000,1.0,2.0,Valor del contrato,Profesional claro,{'url': 'https://community.secop.gov.co/Public...
2,OSCAR MAURICIO REINA GARCIA,Alfonso Eljach,CONTRATO 0738-21,9707321.0,55169729.0,39900000,1.0,6.0,Valor del contrato,Profesional claro,{'url': 'https://community.secop.gov.co/Public...
3,ELKIN MAURICIO,Alfonso Eljach,CONTRATO 3363-21,9702812.0,15937602.0,11900000,1.0,2.0,Valor del contrato,Profesional claro,{'url': 'https://community.secop.gov.co/Public...
4,Carlos Ariel Urzola Núñez,Alfonso Eljach,CONTRATO 2611-21,9691949.0,33113097.0,24500000,1.0,3.0,Valor del contrato,Profesional claro,{'url': 'https://community.secop.gov.co/Public...
5,Edwin De la Cruz,Alfonso Eljach,CONTRATO 3070-21,9685139.0,23544687.0,17500000,1.0,2.0,Valor del contrato,Profesional claro,{'url': 'https://community.secop.gov.co/Public...
6,Carlos Ariel Urzola Núñez,Alfonso Eljach,CO1.PCCNTR.2561097,9624824.0,28773292.0,21000000,1.0,3.0,Valor del contrato,Profesional claro,{'url': 'https://community.secop.gov.co/Public...
7,Edwin De la Cruz,Alfonso Eljach,CONTRATO 1595-21,9581069.0,28642486.0,21000000,1.0,3.0,Valor del contrato,Profesional claro,{'url': 'https://community.secop.gov.co/Public...
8,Fayver Libardo Carrillo Rubio,Alfonso Eljach,CONTRATO 1439-21,9572230.0,19182195.0,14000000,1.0,2.0,Valor del contrato,Profesional claro,{'url': 'https://community.secop.gov.co/Public...
9,OSCAR MAURICIO REINA GARCIA,Alfonso Eljach,CONTRATO 3084-21,9556003.0,22916828.0,17033333,1.0,2.0,Valor del contrato,Profesional claro,{'url': 'https://community.secop.gov.co/Public...


## Análisis de sensibilidad

¿Las conclusiones dependen de haber usado el criterio estricto de CPS? Se repite la comparación
principal con el universo **ampliado**. Si el signo y el orden de magnitud se mantienen, el
hallazgo es robusto.


In [18]:
# 18. Sensibilidad: CPS estricto vs ampliado en la ventana alineada

def comparar(df, etiqueta):
    sub = df[
        df["ventana_alineada_gob_16_33"]
        & df["valor_mensual_real"].notna()
        & df["duracion_meses_exacta"].ge(1)
    ]
    r = (
        sub.groupby("alcalde")
        .agg(
            cps=("contrato_llave", "nunique"),
            personas=("proveedor_llave", "nunique"),
            vm_real_mediano=("valor_mensual_real", "median")
        )
        .reset_index()
    )
    r.insert(0, "universo", etiqueta)
    return r


sensibilidad = pd.concat(
    [comparar(cps, "CPS estricto"), comparar(cps_ampliado, "CPS ampliado")],
    ignore_index=True
)

piv = sensibilidad.pivot(index="universo", columns="alcalde", values="vm_real_mediano")
sensibilidad_brecha = (
    (piv["Jonathan Vasquez"] / piv["Alfonso Eljach"] - 1) * 100
).round(1).rename("brecha_real_%")

display(sensibilidad.round(0))
print("\nBrecha del valor mensual real (Jonathan vs Alfonso) según el universo:")
display(sensibilidad_brecha)
print("\nSi la brecha mantiene el signo en ambos universos, la conclusión es robusta.")

,universo,alcalde,cps,personas,vm_real_mediano
0,CPS estricto,Alfonso Eljach,7305,3603,3419675.0
1,CPS estricto,Jonathan Vasquez,7552,4225,2923020.0
2,CPS ampliado,Alfonso Eljach,7308,3604,3419846.0
3,CPS ampliado,Jonathan Vasquez,7554,4227,2924949.0



Brecha del valor mensual real (Jonathan vs Alfonso) según el universo:


universo
CPS ampliado   -14.5
CPS estricto   -14.5
Name: brecha_real_%, dtype: float64


Si la brecha mantiene el signo en ambos universos, la conclusión es robusta.


In [19]:
# 19. Guardar todas las tablas del análisis

tablas = {
    "04_serie_mensual_cps.csv": serie_mensual,
    "04_tandas_contratacion.csv": tandas,
    "04_resumen_anual.csv": resumen_anual,
    "04_volumen_personas_ventana_alineada.csv": resumen_b,
    "04_duracion_percentiles.csv": percentiles_duracion.reset_index(),
    "04_duracion_distribucion_rangos.csv": distribucion_duracion,
    "04_valores_reales_ventana_alineada.csv": resumen_valores,
    "04_recurrencia_distribucion.csv": distribucion_recurrencia,
    "04_recurrencia_indicadores.csv": resumen_recurrencia,
    "04_renovaciones_distribucion.csv": distribucion_renovaciones,
    "04_contratistas_compartidos.csv": detalle_compartidos,
    "04_electoral_por_anio.csv": electoral,
    "04_ranking_personas.csv": ranking_personas,
    "04_ranking_contratos_valor_mensual.csv": ranking_contratos_vm,
    "04_sensibilidad_universos.csv": sensibilidad
}

for nombre, tabla in tablas.items():
    tabla.to_csv(RUTA_TABLAS / nombre, index=False, encoding="utf-8-sig")

# Base de personas, insumo para el cuaderno de visualización
ranking_personas.to_parquet(RUTA_PROCESADOS / "04_personas_cps_alcaldia.parquet", index=False)
cps.to_parquet(RUTA_PROCESADOS / "04_cps_alcaldia_analitico.parquet", index=False)

print(f"Guardadas {len(tablas)} tablas en entregables/tablas")
print("Guardadas 2 bases en datos/procesados")

Guardadas 15 tablas en entregables/tablas
Guardadas 2 bases en datos/procesados


In [20]:
# 20. Control final

print("=" * 72)
print("CUADERNO 04 COMPLETADO")
print("=" * 72)
print(f"Universo CPS estricto Alcaldía : {len(cps):,} contratos")
print(f"Personas naturales distintas   : {cps['proveedor_llave'].nunique():,}")
print(f"Contratistas en ambos gobiernos: {len(compartidos):,}")
print(f"Ventana de comparación         : meses de gobierno 16-33 ({N_MESES_ALINEADA} meses)")
print()
print("Reglas aplicadas:")
print("  - Dinero siempre en pesos constantes de 2025")
print("  - Volúmenes solo en ventana alineada por mes de gobierno")
print("  - Sin imputación de fechas, valores ni subtipo")
print("  - Contratos simultáneos separados de las renovaciones")
print()
print("Siguiente paso sugerido: cuaderno 05 de visualización y narrativa periodística.")

CUADERNO 04 COMPLETADO
Universo CPS estricto Alcaldía : 26,323 contratos
Personas naturales distintas   : 8,724
Contratistas en ambos gobiernos: 1,589
Ventana de comparación         : meses de gobierno 16-33 (18 meses)

Reglas aplicadas:
  - Dinero siempre en pesos constantes de 2025
  - Volúmenes solo en ventana alineada por mes de gobierno
  - Sin imputación de fechas, valores ni subtipo
  - Contratos simultáneos separados de las renovaciones

Siguiente paso sugerido: cuaderno 05 de visualización y narrativa periodística.
